In [13]:
"""
analyze_methods.py
------------------
Reads a CSV with mapper evaluation results and prints a comparison
table of LLM vs. MultiStep performance, averaged across datasets.

Usage:
    python analyze_methods.py results.csv
    python analyze_methods.py  # uses embedded sample data
"""

import pandas as pd

# ── metric definitions ────────────────────────────────────────────────────────
METRICS = {
    "agg_viol_precision": "Viol. Precision",
    "agg_viol_recall": "Viol. Recall",
    "agg_viol_f1": "Viol. F1",
    "constraint_exact_precision": "Constr. Precision",
    "constraint_exact_recall": "Constr. Recall",
    "constraint_exact_f1": "Constr. F1",
    "pairwise_viol_f1_mean": "Pairwise Viol. F1",
    "runtime_s": "Runtime (s)",
}

QUALITY_METRICS = [
    "agg_viol_f1",
    "constraint_exact_f1",
    "pairwise_viol_f1_mean",
]

def classify_method(variant_name: str) -> str:
    name = variant_name.lower()
    if name.startswith("llm"):
        return "LLM"
    if name.startswith("multistep"):
        return "MultiStep"
    return "Unknown"


def extract_model(variant_name: str) -> str:
    parts = [part.strip() for part in variant_name.split("|")]
    return parts[1] if len(parts) >= 2 else "unknown"


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["method"] = df["variant_name"].apply(classify_method)
    df = df[df["method"].isin(["LLM", "MultiStep"])]
    return df


def per_dataset_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Average each metric per (dataset, method)."""
    cols = ["dataset", "method"] + list(METRICS.keys())
    available = [col for col in cols if col in df.columns]
    return (
        df[available]
        .groupby(["dataset", "method"])
        .mean(numeric_only=True)
        .reset_index()
    )


def overall_summary(per_ds: pd.DataFrame) -> pd.DataFrame:
    """Average the per-dataset means across all datasets."""
    metric_cols = [col for col in METRICS if col in per_ds.columns]
    return per_ds.groupby("method")[metric_cols].mean().reset_index()


def per_model_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Average each model across the full dataset set, penalizing missing datasets."""
    df = df.copy()
    df["model_label"] = df["variant_name"].apply(extract_model)

    metric_cols = [col for col in METRICS if col in df.columns]
    quality_cols = [col for col in QUALITY_METRICS if col in df.columns]
    dataset_names = sorted(df["dataset"].dropna().unique())

    tmp = (
        df[["dataset", "method", "model_label"] + metric_cols]
        .groupby(["dataset", "method", "model_label"])
        .mean(numeric_only=True)
        .reset_index()
    )

    coverage = (
        tmp.groupby(["method", "model_label"])["dataset"]
        .nunique()
        .rename("datasets_present")
        .reset_index()
    )

    completed = []
    for (method, model_label), group in tmp.groupby(["method", "model_label"]):
        group = group.set_index("dataset").reindex(dataset_names)
        group["method"] = method
        group["model_label"] = model_label
        for col in quality_cols:
            group[col] = group[col].fillna(0.0)
        completed.append(group.reset_index())

    full = pd.concat(completed, ignore_index=True)
    per_model = full.groupby(["method", "model_label"])[metric_cols].mean().reset_index()
    per_model = per_model.merge(coverage, on=["method", "model_label"], how="left")
    per_model["datasets_total"] = len(dataset_names)
    return per_model


def winner_marker(llm_val: float, ms_val: float, higher_is_better: bool) -> tuple[str, str]:
    """Return (llm_marker, ms_marker) with ✓ for winner."""
    if higher_is_better:
        llm_wins = llm_val >= ms_val
    else:
        llm_wins = llm_val <= ms_val
    return ("✓", " ") if llm_wins else (" ", "✓")


def print_table(overall: pd.DataFrame):
    llm_row = overall[overall["method"] == "LLM"].iloc[0]
    ms_row = overall[overall["method"] == "MultiStep"].iloc[0]

    col_w = 22
    print()
    print("=" * 72)
    print("  LLM vs. MultiStep — average across all datasets")
    print("=" * 72)
    header = f"{'Metric':<26} {'LLM':>{col_w}} {'MultiStep':>{col_w}}   {'Winner'}"
    print(header)
    print("-" * 72)

    for col, label in METRICS.items():
        if col not in overall.columns:
            continue
        lv = llm_row[col]
        mv = ms_row[col]
        higher_better = col != "runtime_s"
        lm, mm = winner_marker(lv, mv, higher_better)

        if col == "runtime_s":
            lv_str = f"{lv:>10.1f} s"
            mv_str = f"{mv:>10.1f} s"
        else:
            lv_str = f"{lv:>10.4f}"
            mv_str = f"{mv:>10.4f}"

        diff = abs(lv - mv)
        winner = "LLM" if lm == "✓" else "Multi"
        diff_str = f"+{diff:.4f}" if col != "runtime_s" else f"+{diff:.1f}s"
        print(f"  {label:<24} {lm} {lv_str}   {mm} {mv_str}   {winner} ({diff_str})")

    print("=" * 72)
    print()


def print_model_table(per_model: pd.DataFrame, method: str):
    sub = per_model[per_model["method"] == method].copy()
    if sub.empty:
        return

    key_metrics = ["agg_viol_f1", "constraint_exact_f1", "pairwise_viol_f1_mean", "runtime_s"]
    key_labels = ["Viol. F1", "Constr. F1", "Pairwise F1", "Runtime (s)"]

    sub = sub.sort_values(["datasets_present", "model_label"], ascending=[False, True]).reset_index(drop=True)
    models = sub["model_label"].tolist()

    col_w = 16
    header_models = "".join(f"{model:>{col_w}}" for model in models)
    sep = "─" * (28 + col_w * len(models))

    print(f"\n  ── {method} ──")
    print(f"  {'Metric':<28}" + header_models)
    print("  " + sep)

    coverage_row = f"  {'Coverage':<28}"
    for _, row in sub.iterrows():
        coverage_text = f"{int(row['datasets_present'])}/{int(row['datasets_total'])}"
        coverage_row += f"{coverage_text:>{col_w}}"
    print(coverage_row)

    for col, label in zip(key_metrics, key_labels):
        if col not in sub.columns:
            continue
        vals = sub[col].tolist()
        valid_pairs = [(idx, value) for idx, value in enumerate(vals) if not pd.isna(value)]
        if not valid_pairs:
            continue
        higher_better = col != "runtime_s"
        best_idx = (
            max(valid_pairs, key=lambda pair: pair[1])[0]
            if higher_better
            else min(valid_pairs, key=lambda pair: pair[1])[0]
        )
        row_text = f"  {label:<28}"
        for idx, value in enumerate(vals):
            if pd.isna(value):
                display = "—"
            elif col == "runtime_s":
                display = f"{value:.1f}s"
            else:
                display = f"{value:.4f}"
            marker = " ✓" if idx == best_idx else "  "
            row_text += f"{marker}{display:>{col_w - 2}}"
        print(row_text)

    print("  " + sep)

    quality_cols = [col for col in QUALITY_METRICS if col in sub.columns]
    sub["avg_quality"] = sub[quality_cols].mean(axis=1)
    ranked = sub.sort_values("avg_quality", ascending=False)[["model_label", "datasets_present", "datasets_total", "avg_quality"]]
    print(f"\n  Ranking by avg quality over all {int(sub['datasets_total'].iloc[0])} datasets ({method}):")
    for rank, (_, row) in enumerate(ranked.iterrows(), 1):
        coverage = f"{int(row['datasets_present'])}/{int(row['datasets_total'])}"
        print(f"    {rank}. {row['model_label']:<30} {row['avg_quality']:.4f}  [{coverage}]")


def print_per_dataset(per_ds: pd.DataFrame):
    print("\n" + "─" * 72)
    print("  Per-dataset breakdown")
    print("─" * 72)
    datasets = per_ds["dataset"].unique()
    for ds in sorted(datasets):
        sub = per_ds[per_ds["dataset"] == ds]
        print(f"\n  [{ds.upper()}]")
        for col, label in METRICS.items():
            if col not in sub.columns:
                continue
            llm_v = sub[sub["method"] == "LLM"][col].values
            ms_v = sub[sub["method"] == "MultiStep"][col].values
            lv = llm_v[0] if len(llm_v) else float("nan")
            mv = ms_v[0] if len(ms_v) else float("nan")
            if pd.isna(lv) and pd.isna(mv):
                continue
            higher_better = col != "runtime_s"
            lm, mm = winner_marker(lv, mv, higher_better)
            fmt = ".1f" if col == "runtime_s" else ".4f"
            suffix = "s" if col == "runtime_s" else ""
            print(f"    {label:<24} LLM {lm} {lv:{fmt}}{suffix}   Multi {mm} {mv:{fmt}}{suffix}")
    print()

In [14]:
path = 'results_summary_all.csv'  # default path if not supplied on command line
df = load_data(path)


df["model_label"] = df["variant_name"].apply(extract_model)

per_ds    = per_dataset_summary(df)
overall   = overall_summary(per_ds)
per_model = per_model_summary(df)

print_table(overall)
print("\n" + "=" * 72)
print("  Best model per method (avg across all datasets)")
print("=" * 72)
print_model_table(per_model, "LLM")
print_model_table(per_model, "MultiStep")
print_per_dataset(per_ds)


  LLM vs. MultiStep — average across all datasets
Metric                                        LLM              MultiStep   Winner
------------------------------------------------------------------------
  Viol. Precision                0.7685   ✓     0.8787   Multi (+0.1102)
  Viol. Recall             ✓     0.7646         0.6856   LLM (+0.0789)
  Viol. F1                       0.6203   ✓     0.6973   Multi (+0.0770)
  Constr. Precision              0.6395   ✓     0.7598   Multi (+0.1204)
  Constr. Recall                 0.5553   ✓     0.6058   Multi (+0.0505)
  Constr. F1                     0.5877   ✓     0.6603   Multi (+0.0726)
  Pairwise Viol. F1              0.5861   ✓     0.6358   Multi (+0.0496)
  Runtime (s)              ✓       36.4 s          307.4 s   LLM (+270.9s)


  Best model per method (avg across all datasets)

  ── LLM ──
  Metric                                 gpt-5      gpt-5-mini   mistral-small   mistral-large   llama-3.3-70b
  ────────────────────────────────